In [ ]:
# %%
import pandas as pd
import numpy as np

import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

az.style.use("arviz-whitegrid")

In [ ]:
# %%
import pandas as pd

DATA_PATH = "../data/raw/DatasetIcotMond.xlsx"
df_raw = pd.read_excel(DATA_PATH)

df_raw.columns = df_raw.columns.str.strip()

print("Raw dataset shape:", df_raw.shape)

In [ ]:
# %%
GAIT_COLS = [
    "ID",
    "Gait Speed",
    "Stride Length",
    "Cadence",
    "Double Support"
]

missing = [c for c in GAIT_COLS if c not in df_raw.columns]
assert len(missing) == 0, f"Missing columns: {missing}"

df_gait = df_raw[GAIT_COLS].copy()

In [ ]:
# %%
df_subject = (
    df_gait
    .groupby("ID", as_index=False)
    .mean(numeric_only=True)
)

print("Subject-level dataset shape:", df_subject.shape)
display(df_subject.head())

In [ ]:
# %%
df_subject = df_subject.rename(columns={
    "Gait Speed": "gait_speed",
    "Stride Length": "stride_length",
    "Cadence": "cadence",
    "Double Support": "double_support"
})

display(df_subject.describe())

In [ ]:
# %%
OUT_PATH = "../data/processed/subject_level_gait_metrics.csv"
df_subject.to_csv(OUT_PATH, index=False)

print(f"Saved subject-level gait metrics to {OUT_PATH}")

In [ ]:
df = pd.read_csv("../data/processed/subject_level_gait_metrics.csv")

In [ ]:
# %%
DATA_PATH = "../data/processed/subject_level_gait_metrics.csv"
df = pd.read_csv(DATA_PATH)

REQUIRED_COLS = [
    "gait_speed",
    "stride_length",
    "cadence",
    "double_support"
]

missing = [c for c in REQUIRED_COLS if c not in df.columns]
assert len(missing) == 0, f"Missing columns: {missing}"

df = df[REQUIRED_COLS].dropna().reset_index(drop=True)

print("Number of subjects:", df.shape[0])
display(df.describe())

In [ ]:
# %%
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(df.values)

X = pd.DataFrame(X, columns=df.columns)

In [ ]:
# %%
coords = {"subject": np.arange(X.shape[0])}

with pm.Model(coords=coords) as measurement_model:

    # Latent motor severity (identified scale)
    severity = pm.Normal(
        "severity",
        mu=0,
        sigma=1,
        dims="subject"
    )

    # Factor loadings (constrained positive)
    lambda_gs = pm.HalfNormal("lambda_gait_speed", sigma=1)
    lambda_sl = pm.HalfNormal("lambda_stride_length", sigma=1)
    lambda_cad = pm.HalfNormal("lambda_cadence", sigma=1)
    lambda_ds = pm.HalfNormal("lambda_double_support", sigma=1)

    # Intercepts
    alpha_gs = pm.Normal("alpha_gait_speed", 0, 1)
    alpha_sl = pm.Normal("alpha_stride_length", 0, 1)
    alpha_cad = pm.Normal("alpha_cadence", 0, 1)
    alpha_ds = pm.Normal("alpha_double_support", 0, 1)

    # Residual variances
    sigma_gs = pm.HalfNormal("sigma_gait_speed", 1)
    sigma_sl = pm.HalfNormal("sigma_stride_length", 1)
    sigma_cad = pm.HalfNormal("sigma_cadence", 1)
    sigma_ds = pm.HalfNormal("sigma_double_support", 1)

    # Likelihoods
    pm.Normal(
        "gait_speed_obs",
        mu=alpha_gs + lambda_gs * severity,
        sigma=sigma_gs,
        observed=X["gait_speed"].values,
        dims="subject"
    )

    pm.Normal(
        "stride_length_obs",
        mu=alpha_sl + lambda_sl * severity,
        sigma=sigma_sl,
        observed=X["stride_length"].values,
        dims="subject"
    )

    pm.Normal(
        "cadence_obs",
        mu=alpha_cad + lambda_cad * severity,
        sigma=sigma_cad,
        observed=X["cadence"].values,
        dims="subject"
    )

    pm.Normal(
        "double_support_obs",
        mu=alpha_ds + lambda_ds * severity,
        sigma=sigma_ds,
        observed=X["double_support"].values,
        dims="subject"
    )

In [ ]:
# %%
with measurement_model:
    trace = pm.sample(
        draws=2000,
        tune=2000,
        chains=4,
        target_accept=0.9,
        random_seed=42
    )

In [ ]:
# %%
summary = az.summary(
    trace,
    var_names=[
        "lambda_gait_speed",
        "lambda_stride_length",
        "lambda_cadence",
        "lambda_double_support",
        "sigma_gait_speed",
        "sigma_stride_length",
        "sigma_cadence",
        "sigma_double_support"
    ],
    round_to=2
)

display(summary)

In [ ]:
# %%
rhat_max = az.rhat(trace).to_array().max().values
ess_min = az.ess(trace).to_array().min().values

print("Max R-hat:", np.round(rhat_max, 3))
print("Min ESS:", int(ess_min))

In [ ]:
# %%
with measurement_model:
    ppc = pm.sample_posterior_predictive(trace, random_seed=42)

az.plot_ppc(ppc, group="posterior", num_pp_samples=50)
plt.show()

In [ ]:
with measurement_model:
    ppc = pm.sample_posterior_predictive(trace, random_seed=42)

ax = az.plot_ppc(ppc, group="posterior", num_pp_samples=50)

# Rimuove la griglia da tutti i pannelli
for a in np.ravel(ax):
    a.grid(False)
    a.spines["top"].set_visible(False)
    a.spines["right"].set_visible(False)

plt.show()

In [ ]:
# =========================================
# Posterior predictive checks — 2x2 layout, Nature-tier styling
# =========================================

import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl

# -------------------------------
# Font: Times New Roman (Nature-compliant)
# -------------------------------
mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman"]
mpl.rcParams["axes.labelsize"] = 12
mpl.rcParams["xtick.labelsize"] = 10
mpl.rcParams["ytick.labelsize"] = 10
mpl.rcParams["legend.fontsize"] = 11
mpl.rcParams["axes.titlesize"] = 12

# -------------------------------
# Posterior predictive sampling
# -------------------------------
with measurement_model:
    ppc = pm.sample_posterior_predictive(trace, random_seed=42)

# -------------------------------
# Force 2x2 layout
# -------------------------------
fig, axes = plt.subplots(
    2, 2,
    figsize=(10, 8),
    constrained_layout=True
)

# Flatten axes for ArviZ compatibility
axes_flat = axes.flatten()

az.plot_ppc(
    ppc,
    group="posterior",
    num_pp_samples=50,
    ax=axes_flat
)

# -------------------------------
# Aesthetic cleanup (Nature-style)
# -------------------------------
for ax in axes_flat:
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Improve tick appearance
    ax.tick_params(axis="both", which="major", length=4, width=1)

    # Make legend readable
    leg = ax.get_legend()
    if leg is not None:
        leg.set_frame_on(False)

plt.show()

In [ ]:
# ============================================================
# Supplementary Table S3
# Robust extraction of Bayesian measurement model parameters
# ============================================================

import arviz as az
import pandas as pd

# ------------------------------------------------------------
# 1. Ensure InferenceData object exists (VERSION-SAFE)
# ------------------------------------------------------------

if "idata" in globals():
    print("InferenceData 'idata' found.")

elif "trace" in globals():
    # Case 1: PyMC >=4 already returns InferenceData
    if hasattr(trace, "posterior"):
        print("trace is already an InferenceData object.")
        idata = trace

    # Case 2: PyMC3 MultiTrace
    else:
        print("Converting PyMC3 MultiTrace -> InferenceData")
        idata = az.from_pymc3(trace)

else:
    raise NameError(
        "Neither 'idata' nor 'trace' found.\n"
        "Run the Bayesian sampling cell before generating Table S2."
    )

# ------------------------------------------------------------
# 2. Parameters to report (loadings + residual variances)
# ------------------------------------------------------------
params = [
    "lambda_gait_speed",
    "lambda_stride_length",
    "lambda_cadence",
    "lambda_double_support",
    "sigma_gait_speed",
    "sigma_stride_length",
    "sigma_cadence",
    "sigma_double_support"
]

# ------------------------------------------------------------
# 3. ArviZ summary (94% HDI – NPJ standard)
# ------------------------------------------------------------
summary = az.summary(
    idata,
    var_names=params,
    hdi_prob=0.94,
    round_to=3
)

# ------------------------------------------------------------
# 4. Format Supplementary Table S2
# ------------------------------------------------------------
table_s2 = (
    summary[[
        "mean",
        "sd",
        "hdi_3%",
        "hdi_97%",
        "ess_bulk",
        "r_hat"
    ]]
    .reset_index()
    .rename(columns={
        "index": "Parameter",
        "mean": "Mean",
        "sd": "SD",
        "hdi_3%": "HDI 3%",
        "hdi_97%": "HDI 97%",
        "ess_bulk": "ESS",
        "r_hat": "R-hat"
    })
)

# ------------------------------------------------------------
# 5. Human-readable labels (journal-ready)
# ------------------------------------------------------------
label_map = {
    "lambda_gait_speed": "Loading: Gait speed",
    "lambda_stride_length": "Loading: Stride length",
    "lambda_cadence": "Loading: Cadence",
    "lambda_double_support": "Loading: Double support",
    "sigma_gait_speed": "Residual SD: Gait speed",
    "sigma_stride_length": "Residual SD: Stride length",
    "sigma_cadence": "Residual SD: Cadence",
    "sigma_double_support": "Residual SD: Double support"
}

table_s2["Parameter"] = table_s2["Parameter"].map(label_map)

# ------------------------------------------------------------
# 6. Display + save
# ------------------------------------------------------------
print("\n=== Supplementary Table S3 ===\n")
display(table_s2)